In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str # This custom state keeps track of the user's favourite colour

## Write to state

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

# The agent would still have to access the state via the ToolRuntime

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [7]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='2e26172c-dab6-4882-a065-fc79add69916'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_626525': 'EnMKcQFpFH0TlB0t55JAqu9Xrp6DpLL+z63bqrFfwuD9Bm/7Ks+rVuD5yq7vtQi+hngzseckmwbALvU2Xcohb2X1xLjZe6rDX08a0d0AAksJ7w4Etrah3UatNvdoLyDfo73LUtifPceiIAALjDf1BryRrER4'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f645-b015-7943-8cf8-e52a51342d8b-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_626525', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 22, 'total_tokens': 91, 'input_toke

Notice that the output now has the `favourite_colour` state instead of just messages.

In [8]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='46d7d6c9-10cc-4347-89e5-6d6e1ea5d06b'),
              AIMessage(content=[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you doing today?", 'extras': {'signature': 'EnMKcQFpFH0Tay5VoAsGkJage4lcoFktkbHyDkt17RcQq4QJC997WkKqlqEsB9BbFfjzOIq/GGk1gTpW2GgX4pndSFeFggXUvr059vRcd0QrBz8pCyvr7rY4YhmgF5RZofKo2aqWvFYHh7ws3jDlJ7fYYqxk'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f646-cd70-74e3-9df0-f5d9e15cee25-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 17, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}})]}


You can also pass updates to the state yourself while invoking the agent, unlike how you update the message content.

## Read state

In [9]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='faf65292-c7cb-4000-9548-992932bc8746'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_489806': 'EnMKcQFpFH0TtNUvKrymElUQSI4X116uGp6snbqX0AMf/KNT0aylgsoz/h70ix6mZySl9StUZWTd5P2aPMS/qaljcfH67hnB6LykfJu57muBqmOa1vSGxcxRtDJeSWOgN9z1Kx25+xpMz/xfLskfDjDb3HSF'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f648-1525-74f2-98eb-b8e6544715c0-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_489806', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input_tok

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='faf65292-c7cb-4000-9548-992932bc8746'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_489806': 'EnMKcQFpFH0TtNUvKrymElUQSI4X116uGp6snbqX0AMf/KNT0aylgsoz/h70ix6mZySl9StUZWTd5P2aPMS/qaljcfH67hnB6LykfJu57muBqmOa1vSGxcxRtDJeSWOgN9z1Kx25+xpMz/xfLskfDjDb3HSF'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f648-1525-74f2-98eb-b8e6544715c0-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_489806', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input_tok